# TomatoSense — Web App (Frontend UI)
### IT2011 — Simple frontend to check the trained sentiment model



## 0. Setup



In [5]:
!pip install -q datasets transformers torch flask pyngrok accelerate

import torch
import numpy as np
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", device)

ds = load_dataset("cornell-movie-review-data/rotten_tomatoes")
print({k: len(v) for k, v in ds.items()})

Running on: cpu
{'train': 8530, 'validation': 1066, 'test': 1066}


## 1. Fine-Tune DistilBERT (the Group's Selected Best Model)



In [6]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=64)

train_tok = ds["train"].map(tokenize, batched=True)
test_tok  = ds["test"].map(tokenize, batched=True)

distilbert_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

training_args = TrainingArguments(
    output_dir="./webapp_model",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=1.5e-5,
    weight_decay=0.2,
    logging_steps=100,
    save_strategy="no",
    report_to="none",
)
trainer = Trainer(model=distilbert_model, args=training_args, train_dataset=train_tok)
trainer.train()

distilbert_model.to(device)
distilbert_model.eval()
print("DistilBERT fine-tuned and ready.")

Using device: cpu


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
100,0.582920


KeyboardInterrupt: 

## 2. Measure the Model's Real Test Accuracy


In [ ]:
test_out = trainer.predict(test_tok)
test_preds = np.argmax(test_out.predictions, axis=1)
TEST_ACC = accuracy_score(ds["test"]["label"], test_preds)
TEST_F1  = f1_score(ds["test"]["label"], test_preds, average="weighted")
print(f"Test accuracy: {TEST_ACC:.4f}   |   weighted F1: {TEST_F1:.4f}")
print("(The group notebook reports 0.8377 for this model; small differences between runs are normal.)")

## 2. Prediction Function



In [ ]:
def predict_sentiment(review_text):
    inputs = tokenizer(review_text, return_tensors="pt", truncation=True, padding=True, max_length=64)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = distilbert_model(**inputs).logits
    proba = torch.softmax(logits, dim=1)[0]
    pred = int(torch.argmax(proba).item())
    return {
        "label": "Positive" if pred == 1 else "Negative",
        "confidence": round(float(proba[pred].item()) * 100, 1),
        "positive_prob": round(float(proba[1].item()) * 100, 1),
        "negative_prob": round(float(proba[0].item()) * 100, 1),
    }

# Quick self-test: clear, negative, and tricky mixed reviews
for t in ["This film was a masterpiece from start to finish.",
          "A tedious, poorly written mess that wastes its talented cast.",
          "The visuals are gorgeous, but the story is a complete disaster."]:
    print(t, "->", predict_sentiment(t))

## 3. The Web App (Flask + Custom Frontend)


In [ ]:
from flask import Flask, request, jsonify, Response

app = Flask(__name__)

PAGE_HTML = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>TomatoSense - Sentiment Checker</title>
<style>
  :root {
    --wine:#531A22; --gold:#D8A838; --gold-deep:#B9862A; --green:#2E9E6D; --red:#C1272D;
    --cream:#F4F1EA; --grey:#5B5147; --card:#FFFFFF; --line:#E4DDD1;
  }
  * { box-sizing:border-box; }
  body { margin:0; font-family:-apple-system,"Segoe UI",Roboto,Arial,sans-serif; background:var(--cream);
         color:var(--wine); min-height:100vh; display:flex; align-items:center; justify-content:center; padding:24px; }
  .card { background:var(--card); max-width:580px; width:100%; border-radius:16px; border:1px solid var(--line);
          box-shadow:0 10px 40px rgba(83,26,34,.12); padding:34px 34px 26px; }
  .brand { font-size:13px; font-weight:700; letter-spacing:2px; color:var(--gold-deep); text-transform:uppercase; }
  h1 { margin:6px 0 4px; font-size:26px; }
  .subtitle { color:var(--grey); font-size:14px; margin-bottom:10px; }
  .model-tag { display:inline-block; background:#FBEEE9; color:var(--gold-deep); font-size:11.5px; font-weight:700;
               padding:4px 12px; border-radius:20px; margin-bottom:18px; }
  textarea { width:100%; min-height:110px; border-radius:10px; border:1.5px solid var(--line); padding:14px;
             font-size:15px; font-family:inherit; resize:vertical; color:var(--wine); }
  textarea:focus { outline:none; border-color:var(--gold); }
  .row { display:flex; align-items:center; justify-content:space-between; margin-top:12px; flex-wrap:wrap; gap:8px; }
  .examples { font-size:12.5px; color:var(--grey); }
  .examples button { all:unset; color:var(--gold-deep); cursor:pointer; text-decoration:underline; margin-left:10px; font-size:12.5px; }
  .count { font-size:12px; color:var(--grey); }
  #submitBtn { margin-top:14px; background:var(--wine); color:#fff; border:none; padding:12px 26px; border-radius:8px;
               font-size:15px; font-weight:600; cursor:pointer; transition:background .15s; }
  #submitBtn:hover { background:#6E2530; }
  #submitBtn:disabled { opacity:.6; cursor:not-allowed; }
  .hint { font-size:11.5px; color:var(--grey); margin-left:10px; }
  .loading,.error { display:none; font-size:14px; margin-top:16px; }
  .loading { color:var(--grey); }
  .error { color:var(--red); font-size:13.5px; }
  .result { margin-top:24px; display:none; text-align:center; padding:22px; border-radius:12px; background:#FAF7F1; border:1px solid var(--line); }
  .badge { display:inline-block; padding:10px 28px; border-radius:24px; font-weight:700; font-size:20px; color:#fff; }
  .badge.positive { background:var(--green); } .badge.negative { background:var(--red); }
  .bar { height:10px; border-radius:6px; background:var(--red); margin:18px auto 6px; overflow:hidden; max-width:360px; }
  .bar > div { height:100%; background:var(--green); width:50%; transition:width .4s; }
  .bar-labels { display:flex; justify-content:space-between; max-width:360px; margin:0 auto; font-size:12px; color:var(--grey); }
  .confidence { font-size:13.5px; color:var(--grey); margin-top:12px; }
  .note { font-size:12.5px; color:var(--gold-deep); margin-top:8px; display:none; }
  .footer { margin-top:18px; font-size:11.5px; color:var(--grey); text-align:center; }
</style>
</head>
<body>
  <div class="card">
    <div class="brand">TomatoSense</div>
    <h1>Movie Review Sentiment Checker</h1>
    <div class="subtitle">Type a movie review to see whether it reads as positive or negative.</div>
    <div class="model-tag">Model: DistilBERT (group's final model) | __ACC__ test accuracy</div>

    <textarea id="reviewInput" maxlength="600" placeholder="e.g. This film was a masterpiece from start to finish."></textarea>
    <div class="row">
      <div class="examples">Try:
        <button onclick="fillExample(0)">positive</button>
        <button onclick="fillExample(1)">negative</button>
        <button onclick="fillExample(2)">tricky mixed</button>
      </div>
      <div class="count" id="count">0 / 600</div>
    </div>
    <button id="submitBtn" onclick="checkSentiment()">Check Sentiment</button>
    <span class="hint">or press Ctrl + Enter</span>

    <div class="loading" id="loading">Running model...</div>
    <div class="error" id="error"></div>

    <div class="result" id="result">
      <span class="badge" id="badge"></span>
      <div class="bar"><div id="barFill"></div></div>
      <div class="bar-labels"><span>Negative</span><span>Positive</span></div>
      <div class="confidence" id="conf"></div>
      <div class="note" id="note">Low confidence: this review may be mixed or sarcastic.</div>
    </div>
    <div class="footer">Trained on the Rotten Tomatoes review dataset. Short English reviews work best.</div>
  </div>

<script>
const examples = [
  "An absolute masterpiece. The acting was brilliant and the plot kept me on the edge of my seat.",
  "A tedious, poorly written mess that wastes its talented cast.",
  "The visuals are gorgeous, but the story is a complete disaster."
];
const input = document.getElementById("reviewInput");
function fillExample(i) { input.value = examples[i]; updateCount(); }
function updateCount() { document.getElementById("count").textContent = input.value.length + " / 600"; }
input.addEventListener("input", updateCount);
input.addEventListener("keydown", e => { if (e.ctrlKey && e.key === "Enter") checkSentiment(); });

async function checkSentiment() {
  const text = input.value.trim();
  const btn = document.getElementById("submitBtn"), loading = document.getElementById("loading");
  const result = document.getElementById("result"), errorBox = document.getElementById("error");
  errorBox.style.display = "none";
  if (!text) { errorBox.textContent = "Please type a review first."; errorBox.style.display = "block"; return; }

  btn.disabled = true; loading.style.display = "block"; result.style.display = "none";
  try {
    const resp = await fetch("/predict", { method:"POST", headers:{ "Content-Type":"application/json" }, body: JSON.stringify({ text }) });
    if (!resp.ok) throw new Error("Server error");
    const d = await resp.json();
    const badge = document.getElementById("badge");
    badge.textContent = d.label;
    badge.className = "badge " + d.label.toLowerCase();
    document.getElementById("barFill").style.width = d.positive_prob + "%";
    document.getElementById("conf").textContent = "Confidence: " + d.confidence + "%  (Positive " + d.positive_prob + "% / Negative " + d.negative_prob + "%)";
    document.getElementById("note").style.display = d.confidence < 70 ? "block" : "none";
    result.style.display = "block";
  } catch (e) {
    errorBox.textContent = "Could not reach the model. Check that the Colab cell is still running.";
    errorBox.style.display = "block";
  } finally {
    btn.disabled = false; loading.style.display = "none";
  }
}
</script>
</body>
</html>"""

@app.route("/")
def home():
    return Response(PAGE_HTML.replace("__ACC__", f"{TEST_ACC:.1%}"), mimetype="text/html")

@app.route("/predict", methods=["POST"])
def predict():
    data = request.get_json(silent=True) or {}
    text = str(data.get("text", "")).strip()[:600]
    if not text:
        return jsonify({"error": "empty input"}), 400
    return jsonify(predict_sentiment(text))

print("Flask app defined.")

## 4. Launch the App (Public Link)


In [ ]:
import threading, getpass, os
from pyngrok import ngrok

# 1. Retrieve NGROK token from Secrets or fallback directly to string token
NGROK_TOKEN = "3GzkrQuBxVyBkEDZ8g9mvI0UP4I_3J1fp2orbwDk8mrTUF7g"

try:
    from google.colab import userdata
    # Tries to get 'NGROK_TOKEN' from Colab Secrets (key icon on left sidebar)
    secret_token = userdata.get("NGROK_TOKEN")
    if secret_token:
        NGROK_TOKEN = secret_token
except Exception:
    pass

# 2. Authenticate and reset existing tunnels
ngrok.set_auth_token(NGROK_TOKEN)
ngrok.kill()

# 3. Create public HTTP tunnel to port 5000
tunnel = ngrok.connect(5000)
print("Your TomatoSense app is live at:", tunnel.public_url)

# 4. Start Flask app in background thread
threading.Thread(target=lambda: app.run(port=5000, use_reloader=False), daemon=True).start()